# Stage 1: shared 3D reconstruction with Depth Anything 3 (Kaggle)

Picks keyframes from both videos, runs Depth Anything 3 (nested giant, metric) on all of them in one pass, and saves depth, confidence, sky mask and cameras for every frame in one shared coordinate frame, in metres. Then runs checks before we trust the result: camera convention, metric scale, how well the two sessions agree, a 3D viewer, and coverage of chosen spots.

**Kaggle setup (once):**
1. Upload `a.mp4` (session 0) and `b.mp4` (session 1) as a Kaggle dataset (Create → New Dataset), then add it to this notebook with *Add Input*.
2. Settings panel: **Accelerator → GPU T4 x2** (one GPU is used), **Internet → On** (needed to download the code and model). GPU and Internet both need a phone-verified account.
3. Run all. Outputs land in `/kaggle/working` and appear in the Output panel to download.

## 1. Install
We don't pip-install the repo itself. Its pinned `numpy<2` and `xformers` would replace Kaggle's numpy and torch, so we just put its source folder on the Python path. xformers is optional in the code (there is a pure-torch fallback). We only install the few extra packages the import needs.

In [ ]:
import os; os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # set before CUDA starts
!git clone -q https://github.com/ByteDance-Seed/Depth-Anything-3.git /kaggle/working/da3
!pip install -q addict evo omegaconf plyfile pycolmap trimesh "moviepy==1.0.3"
import sys; sys.path.insert(0, "/kaggle/working/da3/src")
from depth_anything_3.api import DepthAnything3
import torch; print("import ok |", torch.cuda.get_device_name(0))

## 2. Pick keyframes
For each video: score every frame for sharpness (variance of the Laplacian; motion blur gives low values), split the video into `N_PER_VIDEO` equal time windows and keep the sharpest frame in each. `FORCE` adds extra frames from a time range. The first 3 s of video B are where the first-row desk is visible, and the VGGT run missed the speaker's old spot without them.

In [ ]:
import cv2, os, glob, shutil, numpy as np

find = lambda n: sorted(glob.glob(f"/kaggle/input/**/{n}", recursive=True))[0]
VIDEOS = [find("a.mp4"), find("b.mp4")]          # session 0, session 1
print(VIDEOS)
N_PER_VIDEO = 22                                # ~49 frames total fits a T4 with the giant model (67 ran out of memory)
FORCE = {1: (0.0, 3.0, 5)}                      # video -> (start s, end s, extra frames)
SAVE_W = 1008                                   # frames are saved at this width (DA3 downsizes anyway)

def sharpness(f):
    s = 480 / f.shape[1]
    g = cv2.cvtColor(cv2.resize(f, None, fx=s, fy=s), cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(g, cv2.CV_64F).var()

def sharpest_per_window(scores, lo, hi, k):
    return {int(w[np.argmax(scores[w])]) for w in np.array_split(np.arange(lo, hi), k) if len(w)}

shutil.rmtree("/kaggle/working/frames", ignore_errors=True); os.makedirs("/kaggle/working/frames")
paths, vid, fidx, FPS = [], [], [], []
for v, name in enumerate(VIDEOS):
    cap = cv2.VideoCapture(name); assert cap.isOpened(), name
    fps = cap.get(cv2.CAP_PROP_FPS); FPS.append(fps)
    scores = []
    while True:
        ok, f = cap.read()
        if not ok: break
        scores.append(sharpness(f))
    cap.release()
    scores, n = np.array(scores), len(scores)
    keep = sharpest_per_window(scores, 0, n, N_PER_VIDEO)
    if v in FORCE:
        t0, t1, k = FORCE[v]
        keep |= sharpest_per_window(scores, int(t0 * fps), min(int(t1 * fps), n), k)
    cap = cv2.VideoCapture(name)
    for i in range(n):
        ok, f = cap.read()
        if not ok: break
        if i in keep:
            f = cv2.resize(f, (SAVE_W, round(f.shape[0] * SAVE_W / f.shape[1])), interpolation=cv2.INTER_AREA)
            p = f"/kaggle/working/frames/{v}_{i:04d}.jpg"; cv2.imwrite(p, f)
            paths.append(p); vid.append(v); fidx.append(i)
    cap.release()
    print(f"{name}: {n} frames at {fps:.0f} fps, kept {len(keep)}")
vid, fidx = np.array(vid), np.array(fidx)
print(len(paths), "frames total")

Contact sheet of what goes in. Check that video B's first frames show the first row.

In [ ]:
import matplotlib.pyplot as plt
for v in (0, 1):
    sel = [p for p, s in zip(paths, vid) if s == v]
    cols = 10; rows = -(-len(sel) // cols)
    fig, ax = plt.subplots(rows, cols, figsize=(cols * 1.8, rows * 1.1))
    for a in ax.flat: a.axis("off")
    for a, p in zip(ax.flat, sel):
        a.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)); a.set_title(os.path.basename(p)[2:-4], fontsize=7)
    fig.suptitle(f"video {v}"); plt.tight_layout(); plt.show()

## 3. Run Depth Anything 3
All frames from both videos go in **one** pass, so they come out in one coordinate frame with no separate alignment step. The nested model has two branches: an any-view branch (relative depth and poses) and a metric branch. It rescales depth and camera translations to metres (`scale_factor`). Sky pixels get pushed to max depth with high confidence, so we keep the sky mask and filter on it. Windows may be labelled as sky.

On a T4 we force float16 and the memory-efficient attention kernel (see the comment in the cell). Without that it tried to allocate 98 GB. If it happens again, lower `N_PER_VIDEO` in cell 2 and use **Restart & Run All**, because a failed run leaves the model in GPU memory.

In [ ]:
import torch, time
MODEL = "depth-anything/DA3NESTED-GIANT-LARGE-1.1"
PROCESS_RES = 504                     # longest image side fed to the model

# A T4 has no real bfloat16, but recent PyTorch reports it as supported, so DA3 picks it. The
# memory-efficient attention kernel can't run bfloat16 on a T4, and PyTorch silently falls back to the
# naive kernel, which wanted 98 GB for the cross-frame attention. Force float16 and require the efficient kernel.
from torch.nn.attention import sdpa_kernel, SDPBackend
torch.cuda.is_bf16_supported = lambda *a, **k: False

model = DepthAnything3.from_pretrained(MODEL).to("cuda").eval()
torch.cuda.reset_peak_memory_stats(); t0 = time.time()
with sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION]):
    pred = model.inference(paths, process_res=PROCESS_RES)
print(f"{time.time() - t0:.0f} s, peak GPU {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

images = np.asarray(pred.processed_images)                      # (S,H,W,3) uint8
depth = np.asarray(pred.depth, np.float32)                      # (S,H,W) metres
conf = np.asarray(pred.conf, np.float32)
sky = np.asarray(pred.sky, bool) if pred.sky is not None else np.zeros(depth.shape, bool)
E_raw = np.asarray(pred.extrinsics, np.float64)[:, :3, :4]      # README says world-to-camera; checked below
K = np.asarray(pred.intrinsics, np.float64)
S, H, W = depth.shape
print(f"{S} frames at {W}x{H} | is_metric={pred.is_metric} scale_factor={pred.scale_factor}")
print("NaNs in depth:", np.isnan(depth).any(), "(float16 overflow if True)")
print(f"sky pixels: {sky.mean():.1%} | depth median {np.median(depth):.2f} m, 99th pct {np.percentile(depth, 99):.2f} m")
del model; torch.cuda.empty_cache()

## 4. Depth to 3D points, plus pixel filters
Each pixel (u, v) with depth z becomes a camera-space point z·K⁻¹[u,v,1], then a world point via the extrinsics. With world-to-camera extrinsics [R|t]: world = Rᵀ(cam − t).

We drop pixels that are low confidence (bottom 20%), on a depth edge (relative depth gradient ≥ 0.05, which removes the "drag" streaks between foreground and background), or sky.

In [ ]:
CONF_PERCENTILE = 20
EDGE_THRESH = 0.05

u, v = np.meshgrid(np.arange(W), np.arange(H))

def unproject(i, E):
    z = depth[i]
    cam = np.stack([(u - K[i][0, 2]) * z / K[i][0, 0], (v - K[i][1, 2]) * z / K[i][1, 1], z], -1)
    R, t = E[i][:, :3], E[i][:, 3]
    return (cam - t) @ R                                         # row-vector form of R.T @ (cam - t)

def project(X, i, E):
    c = X @ E[i][:, :3].T + E[i][:, 3]
    z = c[..., 2]
    with np.errstate(divide="ignore", invalid="ignore"):
        uu = K[i][0, 0] * c[..., 0] / z + K[i][0, 2]
        vv = K[i][1, 1] * c[..., 1] / z + K[i][1, 2]
    return uu, vv, z

def invert(E):                                                   # [R|t] -> [R.T | -R.T t]
    R, t = E[:, :, :3], E[:, :, 3:]
    Rt = R.transpose(0, 2, 1)
    return np.concatenate([Rt, -Rt @ t], -1)

good = (conf > np.percentile(conf, CONF_PERCENTILE)) & ~sky & (depth > 0)
for i in range(S):
    gy, gx = np.gradient(depth[i])
    good[i] &= np.sqrt(gx**2 + gy**2) / np.maximum(depth[i], 1e-6) < EDGE_THRESH
print(f"pixels kept: {good.mean():.0%}")

## 5. Check: camera convention
We don't trust the README alone. Take good pixels in one frame, unproject them, project them into a nearby frame and check whether the depths agree. Do this assuming world-to-camera, then assuming camera-to-world. The right convention gives high agreement and the wrong one gives near zero. From here on, `E` is world-to-camera either way.

In [ ]:
def agreement(E, pairs, n=20000, rng=np.random.default_rng(0)):
    out = []
    for i, j in pairs:
        ys, xs = np.nonzero(good[i]); k = rng.choice(len(ys), min(n, len(ys)), replace=False)
        X = unproject(i, E)[ys[k], xs[k]]
        uu, vv, z = project(X, j, E)
        ok = (z > 0) & (uu >= 0) & (uu < W) & (vv >= 0) & (vv < H)
        ui, vi = uu[ok].astype(int), vv[ok].astype(int)
        g = good[j][vi, ui]
        if g.sum() < 100: continue
        out.append(np.mean(np.abs(z[ok][g] - depth[j][vi, ui][g]) < 0.05 * z[ok][g]))
    return np.median(out) if out else np.nan

same_video = [(i, i + 2) for i in range(S - 2) if vid[i] == vid[i + 2]]
a_w2c, a_c2w = agreement(E_raw, same_video), agreement(invert(E_raw), same_video)
print(f"same-video pairs agreeing within 5% depth: as world-to-camera {a_w2c:.0%}, as camera-to-world {a_c2w:.0%}")
E = E_raw if a_w2c >= a_c2w else invert(E_raw)
print("using", "world-to-camera as given" if a_w2c >= a_c2w else "INVERTED extrinsics")

## 6. Point cloud, metric sanity, and agreement between sessions
If the scale really is metric, the room should be a few metres across and the camera should move a few cm to tens of cm between keyframes (walking pace). The two sessions' clouds should overlap closely. The median distance from session 1 points to the nearest session 0 point measures how well they line up. This is the noise floor that every detected change has to clear.

In [ ]:
from scipy.spatial import cKDTree
P = np.stack([unproject(i, E) for i in range(S)])               # (S,H,W,3) world points
centres = np.stack([-E[i][:, :3].T @ E[i][:, 3] for i in range(S)])

def voxel_points(mask, size=0.02, min_support=5):
    p, c = P[mask], images[mask]
    key = np.floor(p / size).astype(np.int64)
    _, first, inv, cnt = np.unique(key, axis=0, return_index=True, return_inverse=True, return_counts=True)
    keep = cnt >= min_support
    return p[first[keep]], c[first[keep]]

cloud = {s: voxel_points(good & (vid == s)[:, None, None]) for s in (0, 1)}
allp = np.concatenate([cloud[0][0], cloud[1][0]])
ax = np.linalg.svd(allp - allp.mean(0), full_matrices=False)[2]           # principal axes
ext = np.percentile((allp - allp.mean(0)) @ ax.T, 99, 0) - np.percentile((allp - allp.mean(0)) @ ax.T, 1, 0)
print("room extent along principal axes (m):", np.round(ext, 2))
for s in (0, 1):
    c = centres[vid == s]
    print(f"session {s}: {len(cloud[s][0]):,} voxels | camera step median {np.median(np.linalg.norm(np.diff(c, axis=0), axis=1)):.2f} m, path {np.linalg.norm(np.diff(c, axis=0), axis=1).sum():.1f} m")
d, _ = cKDTree(cloud[0][0]).query(cloud[1][0])
ov = d < 0.5                                                     # ignore parts only session 1 saw
print(f"session 1 -> nearest session 0 point, in the overlap ({ov.mean():.0%} of session 1): "
      f"median {np.median(d[ov])*100:.1f} cm, 75th pct {np.percentile(d[ov], 75)*100:.1f} cm")

## 7. Look at it
Colour by `rgb` or by `session` (blue = 0, red = 1). Big diamonds are cameras. Hover shows coordinates. Write down the speaker's old and new positions, because cell 8 checks them.

In [ ]:
import plotly.graph_objects as go
COLOR_BY = "rgb"          # "rgb" or "session"
MAX_POINTS = 250_000

rng = np.random.default_rng(0)
traces = []
for s, col in ((0, "royalblue"), (1, "crimson")):
    p, c = cloud[s]
    k = rng.choice(len(p), min(len(p), MAX_POINTS // 2), replace=False)
    colour = [f"rgb({r},{g},{b})" for r, g, b in c[k]] if COLOR_BY == "rgb" else col
    traces.append(go.Scatter3d(x=p[k, 0], y=p[k, 1], z=p[k, 2], mode="markers",
                               marker=dict(size=1.5, color=colour), name=f"session {s}"))
    cc = centres[vid == s]
    traces.append(go.Scatter3d(x=cc[:, 0], y=cc[:, 1], z=cc[:, 2], mode="markers+lines",
                               marker=dict(size=3, color=col, symbol="diamond"), line=dict(color=col), name=f"cameras {s}"))
fig = go.Figure(traces)
fig.update_layout(scene=dict(aspectmode="data"), height=800, margin=dict(l=0, r=0, t=0, b=0))
fig.write_html("/kaggle/working/cloud.html"); fig.show()

## 8. Coverage of a spot
For a 3D point, each frame votes: **visible** (observed depth matches), **see-through** (observed surface is well behind it, so the spot is empty in that frame), **occluded** (something in front), or no vote (out of view or filtered pixel). This is the same test the change detector will use, with τ = 0.05·depth + 2 cm. For the speaker's old spot we want session 1 to have see-through votes. Before, it had zero frames in view.

In [ ]:
SPOTS = {                     # fill in from the viewer (metres)
    # "speaker_old": (0.0, 0.0, 0.0),
    # "speaker_new": (0.0, 0.0, 0.0),
}
VOXEL = 0.02

def coverage(p):
    p = np.asarray(p, float)
    res = {0: [0, 0, 0, 0], 1: [0, 0, 0, 0]}                     # in view, visible, see-through, occluded
    best = {0: [], 1: []}
    for i in range(S):
        uu, vv, z = project(p[None], i, E)
        uu, vv, z = uu[0], vv[0], z[0]
        if not (z > 0 and 0 <= uu < W and 0 <= vv < H): continue
        ui, vi = int(uu), int(vv)
        if not good[i][vi, ui]: continue
        r, tau, s = z - depth[i][vi, ui], 0.05 * z + VOXEL, vid[i]
        res[s][0] += 1
        if abs(r) <= tau: res[s][1] += 1; best[s].append((z, i, ui, vi))
        elif r < -tau: res[s][2] += 1; best[s].append((z, i, ui, vi))
        else: res[s][3] += 1
    return res, best

for name, p in SPOTS.items():
    res, best = coverage(p)
    for s in (0, 1):
        print(f"{name} | session {s}: in view {res[s][0]}, visible {res[s][1]}, see-through {res[s][2]}, occluded {res[s][3]}")
    fig, ax = plt.subplots(1, 6, figsize=(16, 2.2))
    shown = sorted(best[0])[:3] + sorted(best[1])[:3]               # closest views per session
    for a in ax: a.axis("off")
    for a, (z, i, ui, vi) in zip(ax, shown):
        a.imshow(images[i]); a.plot(ui, vi, "o", mfc="none", mec="yellow", ms=14, mew=2)
        a.set_title(f"s{vid[i]} frame {fidx[i]} ({z:.1f} m)", fontsize=8)
    fig.suptitle(name); plt.show()

## 9. Save
`extrinsic` is always world-to-camera (OpenCV) [R|t], in metres. `video_id` = session, `frame_idx` = original frame number in that video. Download `recon_da3.npz` and `cloud.html` from the Output panel (right sidebar → Output, or Save Version → Output tab).

In [ ]:
np.savez_compressed("/kaggle/working/recon_da3.npz",
    images=images, depth=depth, conf=conf, sky=sky, extrinsic=E.astype(np.float32), intrinsic=K.astype(np.float32),
    video_id=vid, frame_idx=fidx, fps=np.array(FPS), scale_factor=np.float32(pred.scale_factor or np.nan),
    model=MODEL, process_res=PROCESS_RES)
print(f"{os.path.getsize('/kaggle/working/recon_da3.npz') / 1e6:.0f} MB")
shutil.rmtree("/kaggle/working/da3", ignore_errors=True)       # keep the Output panel tidy
shutil.rmtree("/kaggle/working/frames", ignore_errors=True)
print(os.listdir("/kaggle/working"), "-> download from the Output panel")